# CLIP4CAD-GFA Ablation Study Training

This notebook trains architecture ablations for CLIP4CAD-GFA:

| Ablation | Description | Config Changes |
|----------|-------------|----------------|
| `baseline` | Full model | No changes |
| `no_consistency` | Disable grounding consistency | `lambda_consist_brep=0, lambda_consist_pc=0` |
| `global_only` | Only global contrastive loss | `lambda_local=0, lambda_consist_*=0, lambda_diverse=0` |
| `no_confidence` | Fixed uniform confidence | `use_uniform_confidence=True` |
| `weak_grounding` | Reduced grounding losses | `lambda_local=0.1, lambda_diverse=0.02` |
| `asymmetric_grounding` | Asymmetric consistency weights | `lambda_consist_brep=0.08, lambda_consist_pc=0.02` |

## Asymmetric Grounding
Uses text-anchored consistency with different weights per modality:
- **B-Rep** (`lambda_consist_brep=0.08`): Needs stronger grounding (pure geometry)
- **Point Cloud** (`lambda_consist_pc=0.02`): Needs less (ShapeLLM has multimodal features)

## Features
- Full GFATrainer with hard negative mining
- Proper 2-stage curriculum
- Checkpointing every 5 epochs
- Load data ONCE, train all ablations

## Usage
1. Run Cells 1-3 to load data (~5 min, only once)
2. Run the cell for the ablation you want to train

---
## Cell 1: Imports & Setup

In [20]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, 'D:/Defect_Det/MMCAD/MMCAD')

import gc
import torch
from omegaconf import OmegaConf

from clip4cad.data.gfa_dataset import GFAMappedDataset
from clip4cad.training.gfa_trainer import GFATrainer
from clip4cad.ablations.models import CLIP4CAD_GFA_Ablation
from clip4cad.ablations.configs import get_ablation_config, ABLATION_TYPES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Available ablation types:")
for name, desc in ABLATION_TYPES.items():
    print(f"  - {name}: {desc}")
print(f"\nDevice: {device}")
print("\n" + "="*60)
print("Run Cell 2 to configure paths, then Cell 3 to load data.")
print("="*60)

Available ablation types:
  - baseline: Full model with all losses and learned confidence
  - no_consistency: Disable grounding consistency loss (λ_c=0)
  - global_only: Only global contrastive loss (no local/grounding)
  - no_confidence: Fixed uniform confidence (no learned weighting)

Device: cuda

Run Cell 2 to configure paths, then Cell 3 to load data.


---
## Cell 2: Configuration

In [21]:
# =============================================================================
# PATHS - Update these for your setup
# =============================================================================
PATHS = {
    "data_root": "d:/Defect_Det/MMCAD/data",
    "pc_file": "c:/Users/User/Desktop/pc_embeddings_full.h5",
    "brep_file": "c:/Users/User/Desktop/brep_features.h5",
    "text_file": "c:/Users/User/Desktop/text_embeddings.h5",
    "config_path": "d:/Defect_Det/MMCAD/MMCAD/configs/model/clip4cad_gfa.yaml",
    "output_base": "outputs/ablations",
}

# =============================================================================
# TRAINING CONFIG - Same for all ablations
# =============================================================================
TRAIN_CONFIG = {
    "batch_size": 512,
    "num_workers": 0,  # CRITICAL: 0 because data is in RAM
    "num_epochs_stage1": 15,
    "num_epochs_stage2": 20,
    "learning_rate": 1e-4,
    "save_every": 5,
}

print("Configuration:")
print(f"  Batch size: {TRAIN_CONFIG['batch_size']}")
print(f"  Stage 1: {TRAIN_CONFIG['num_epochs_stage1']} epochs")
print(f"  Stage 2: {TRAIN_CONFIG['num_epochs_stage2']} epochs")
print(f"  Total: {TRAIN_CONFIG['num_epochs_stage1'] + TRAIN_CONFIG['num_epochs_stage2']} epochs")
print(f"  Output: {PATHS['output_base']}")

Configuration:
  Batch size: 512
  Stage 1: 15 epochs
  Stage 2: 20 epochs
  Total: 35 epochs
  Output: outputs/ablations


---
## Cell 3: Load Datasets (RUN ONCE)

**This loads ~204GB into RAM. Takes ~5 min but only needs to run ONCE.**

In [22]:
print("Loading datasets (this takes ~5 min, but only once!)...")
print("=" * 60)

# Train dataset - LOAD TO RAM
print("\n[1/2] Loading TRAIN dataset to RAM...")
train_dataset = GFAMappedDataset(
    data_root=PATHS["data_root"],
    split="train",
    pc_file=PATHS["pc_file"],
    text_file=PATHS["text_file"],
    brep_file=PATHS["brep_file"],
    num_rotations=1,
    load_to_memory=True,
    use_live_text=False,
)
print(f"Train: {len(train_dataset):,} samples in RAM")

# Val dataset - ON DISK (saves RAM)
print("\n[2/2] Loading VAL dataset (on disk)...")
val_dataset = GFAMappedDataset(
    data_root=PATHS["data_root"],
    split="val",
    pc_file=PATHS["pc_file"],
    text_file=PATHS["text_file"],
    brep_file=PATHS["brep_file"],
    num_rotations=1,
    load_to_memory=False,
    use_live_text=False,
)
print(f"Val: {len(val_dataset):,} samples on disk")

print("\n" + "=" * 60)
print("DATASETS READY!")
print("Do NOT re-run this cell unless you restart the kernel!")
print("=" * 60)

Loading datasets (this takes ~5 min, but only once!)...

[1/2] Loading TRAIN dataset to RAM...
  Loading train data to memory (B-Rep + PC + Text)...
    Loading B-Rep (3GB)...


KeyboardInterrupt: 

---
## Helper Function: Train Ablation

This function handles all the setup for training an ablation.

In [ ]:
def train_ablation(ablation_type: str, resume_from: str = None, custom_overrides: dict = None):
    """
    Train a specific ablation using the full GFATrainer.
    
    Args:
        ablation_type: One of 'baseline', 'no_consistency', 'global_only', 'no_confidence', 
                       'weak_grounding', 'asymmetric_grounding'
        resume_from: Optional checkpoint path to resume from
        custom_overrides: Optional dict of config overrides
    """
    print("\n" + "=" * 70)
    print(f"ABLATION: {ablation_type}")
    print(f"Description: {ABLATION_TYPES[ablation_type]}")
    print("=" * 70)
    
    # Merge custom overrides with TRAIN_CONFIG
    overrides = {"training": TRAIN_CONFIG.copy()}
    if custom_overrides:
        for k, v in custom_overrides.items():
            if k == "training":
                overrides["training"].update(v)
            else:
                overrides[k] = v
    
    # Get ablation config
    config = get_ablation_config(
        PATHS["config_path"],
        ablation_type,
        custom_overrides=overrides
    )
    
    # Output directory
    output_dir = Path(PATHS["output_base"]) / ablation_type
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save config
    OmegaConf.save(config, output_dir / "config.yaml")
    
    # Print loss weights (asymmetric consistency)
    print("\nLoss weights:")
    print(f"  lambda_global: {config.training.lambda_global}")
    print(f"  lambda_local: {config.training.lambda_local}")
    print(f"  lambda_consist_brep: {config.training.lambda_consist_brep}")
    print(f"  lambda_consist_pc: {config.training.lambda_consist_pc}")
    print(f"  lambda_diverse: {config.training.lambda_diverse}")
    print(f"  lambda_conf_reg: {config.training.lambda_conf_reg}")
    
    # Create ablation model
    print("\nCreating model...")
    model = CLIP4CAD_GFA_Ablation(config)
    print(f"  Parameters: {model.count_parameters():,}")
    print(f"  Trainable: {model.count_parameters(trainable_only=True):,}")
    
    # Create trainer with full GFATrainer (has hard neg mining, checkpointing, etc.)
    trainer = GFATrainer(
        model=model,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        config=OmegaConf.to_container(config.training),
        device=device,
        output_dir=str(output_dir),
    )
    
    # Resume if specified
    if resume_from:
        trainer.resume(resume_from)
    
    # Train
    print("\nStarting training...")
    trainer.train()
    
    print("\n" + "=" * 70)
    print(f"ABLATION {ablation_type} COMPLETE!")
    print(f"Checkpoints saved to: {output_dir}")
    print("=" * 70)
    
    # Cleanup
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    
    return output_dir

print("Helper function defined. Run the ablation cells below.")

---
---
# ABLATION TRAINING CELLS

Run ONE of the cells below to train that specific ablation.

---

## Ablation 1: BASELINE (Full Model)

Full model with all losses enabled. This is the control condition.

## Ablation 2: NO CONSISTENCY

Disable grounding consistency loss (`lambda_consist=0`).

Tests whether the grounding consistency constraint helps alignment.

In [ ]:
# NO CONSISTENCY: lambda_consist=0
train_ablation("no_consistency")


ABLATION: no_consistency
Description: Disable grounding consistency loss (λ_c=0)

Loss weights:
  lambda_global: 1.0
  lambda_local: 0.5
  lambda_consist: 0.0
  lambda_diverse: 0.2
  lambda_conf_reg: 0.1

Creating model...
  [Ablation] Using LEARNED confidence
  Parameters: 3,168,771
  Trainable: 3,168,771


D:\Defect_Det/MMCAD/MMCAD\clip4cad\training\gfa_trainer.py:179: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler()



Starting training...
CLIP4CAD-GFA Training
Total epochs: 35
Stage 1: 15 epochs (grounding)
Stage 2: 20 epochs (alignment)
Batch size: 512
Learning rate: 0.0001
Checkpoint every: 5 epochs
Trainable parameters: 3,168,771



Epoch 1 (Stage 1):   0%|          | 0/216 [00:00<?, ?it/s]D:\Defect_Det/MMCAD/MMCAD\clip4cad\training\gfa_trainer.py:335: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1 (Stage 1):   0%|          | 1/216 [00:06<23:08,  6.46s/it, loss=4.587, G=6.630, L=6.379, C=0.933, D=0.012, cf=0.000, lr=1.5e-07]


  [Conf] min=0.451 max=0.514 mean=0.483 active=12.0/12


...

Epoch 35 (Stage 2):  93%|█████████▎| 201/216 [02:25<00:10,  1.42it/s, loss=0.420, G=0.402, L=0.002, C=0.984, D=0.015, cf=0.000, lr=1.0e-06]


  [Conf] min=0.007 max=0.993 mean=0.320 active=5.8/12


Validation: 100%|██████████| 33/33 [03:54<00:00,  7.11s/it]



Epoch 35 Summary:
  Train Loss: 0.4398
    Global: 0.4209
    Local: 0.0025
    Consistency: 0.9833
    Diversity: 0.0152
    Conf Floor: 0.0006
  Val Loss: 1.0851
Final model saved: outputs\ablations\no_consistency\clip4cad_gfa_final.pt

Training Complete!

ABLATION no_consistency COMPLETE!
Checkpoints saved to: outputs\ablations\no_consistency


WindowsPath('outputs/ablations/no_consistency')

## Ablation 3: GLOBAL ONLY

Only global contrastive loss, no local/grounding losses.

Tests whether fine-grained grounding helps vs. just global alignment.

In [ ]:
# GLOBAL ONLY: No local/grounding losses
train_ablation("global_only")


ABLATION: global_only
Description: Only global contrastive loss (no local/grounding)

Loss weights:
  lambda_global: 1.0
  lambda_local: 0.0
  lambda_consist: 0.0
  lambda_diverse: 0.0
  lambda_conf_reg: 0.1

Creating model...
  [Ablation] Using LEARNED confidence
  Parameters: 3,168,771
  Trainable: 3,168,771

Starting training...
CLIP4CAD-GFA Training
Total epochs: 35
Stage 1: 15 epochs (grounding)
Stage 2: 20 epochs (alignment)
Batch size: 512
Learning rate: 0.0001
Checkpoint every: 5 epochs
Trainable parameters: 3,168,771



Epoch 1 (Stage 1):   0%|          | 1/216 [00:02<08:32,  2.39s/it, loss=1.370, G=6.502, L=6.299, C=0.974, D=0.008, cf=0.000, lr=1.5e-07]


  [Conf] min=0.432 max=0.495 mean=0.465 active=12.0/12


Epoch 1 (Stage 1):  47%|████▋     | 101/216 [00:38<00:39,  2.93it/s, loss=1.213, G=5.746, L=6.274, C=0.936, D=0.014, cf=0.000, lr=1.6e-05]


  [Conf] min=0.305 max=0.445 mean=0.341 active=12.0/12


...

Epoch 35 (Stage 2):  93%|█████████▎| 201/216 [02:17<00:10,  1.48it/s, loss=0.404, G=0.378, L=4.472, C=0.980, D=0.020, cf=0.001, lr=1.0e-06]


  [Conf] min=0.007 max=0.993 mean=0.299 active=5.4/12


Validation: 100%|██████████| 33/33 [04:01<00:00,  7.31s/it]



Epoch 35 Summary:
  Train Loss: 0.4310
    Global: 0.4047
    Local: 4.4310
    Consistency: 0.9774
    Diversity: 0.0228
    Conf Floor: 0.0008
  Val Loss: 0.7810
Final model saved: outputs\ablations\global_only\clip4cad_gfa_final.pt

Training Complete!

ABLATION global_only COMPLETE!
Checkpoints saved to: outputs\ablations\global_only


WindowsPath('outputs/ablations/global_only')

## Ablation 4: NO CONFIDENCE

Fixed uniform confidence weighting (all slots = 1/K) instead of learned confidence.

Tests whether learned confidence weighting helps.

In [ ]:
# NO CONFIDENCE: Fixed uniform weighting
train_ablation("no_confidence")


ABLATION: no_confidence
Description: Fixed uniform confidence (no learned weighting)

Loss weights:
  lambda_global: 1.0
  lambda_local: 0.5
  lambda_consist: 0.5
  lambda_diverse: 0.2
  lambda_conf_reg: 0.0

Creating model...
  [Ablation] Using UNIFORM confidence (all slots = 1/K)
  Parameters: 3,168,771
  Trainable: 3,168,771

Starting training...
CLIP4CAD-GFA Training
Total epochs: 35
Stage 1: 15 epochs (grounding)
Stage 2: 20 epochs (alignment)
Batch size: 512
Learning rate: 0.0001
Checkpoint every: 5 epochs
Trainable parameters: 3,168,771



Epoch 1 (Stage 1):   0%|          | 1/216 [00:02<10:05,  2.82s/it, loss=5.143, G=6.605, L=6.296, C=1.130, D=0.000, cf=0.217, lr=1.5e-07]


  [Conf] min=0.083 max=0.083 mean=0.083 active=3.0/12


Epoch 1 (Stage 1):  47%|████▋     | 101/216 [01:35<01:48,  1.06it/s, loss=2.552, G=6.342, L=1.912, C=0.437, D=0.001, cf=0.217, lr=1.6e-05]


  [Conf] min=0.083 max=0.083 mean=0.083 active=3.0/12


...

Epoch 35 (Stage 2):  93%|█████████▎| 201/216 [02:23<00:10,  1.39it/s, loss=0.535, G=0.425, L=0.002, C=0.000, D=0.001, cf=0.217, lr=1.0e-06]


  [Conf] min=0.083 max=0.083 mean=0.083 active=3.0/12


Validation: 100%|██████████| 33/33 [03:58<00:00,  7.24s/it]



Epoch 35 Summary:
  Train Loss: 0.5154
    Global: 0.4056
    Local: 0.0023
    Consistency: 0.0002
    Diversity: 0.0014
    Conf Floor: 0.2167
  Val Loss: 1.1436
Final model saved: outputs\ablations\no_confidence\clip4cad_gfa_final.pt

Training Complete!

ABLATION no_confidence COMPLETE!
Checkpoints saved to: outputs\ablations\no_confidence


WindowsPath('outputs/ablations/no_confidence')

In [25]:
import importlib
# Reload the configs module to pick up the new ablation                                                                 import importlib
import clip4cad.ablations.configs
importlib.reload(clip4cad.ablations.configs)
from clip4cad.ablations.configs import get_ablation_config, ABLATION_TYPES

# Train weak_grounding
train_ablation("weak_grounding")

D:\Defect_Det/MMCAD/MMCAD\clip4cad\training\gfa_trainer.py:179: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler()



ABLATION: weak_grounding
Description: Reduced grounding losses (λ_l=0.1, λ_c=0.05, λ_d=0.02)

Loss weights:
  lambda_global: 1.0
  lambda_local: 0.1
  lambda_consist: 0.05
  lambda_diverse: 0.02
  lambda_conf_reg: 0.1

Creating model...
  [Ablation] Using LEARNED confidence
  Parameters: 3,168,771
  Trainable: 3,168,771

Starting training...
CLIP4CAD-GFA Training
Total epochs: 35
Stage 1: 15 epochs (grounding)
Stage 2: 20 epochs (alignment)
Batch size: 512
Learning rate: 0.0001
Checkpoint every: 5 epochs
Trainable parameters: 3,168,771



Epoch 1 (Stage 1):   0%|          | 0/216 [00:00<?, ?it/s]D:\Defect_Det/MMCAD/MMCAD\clip4cad\training\gfa_trainer.py:335: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1 (Stage 1):   0%|          | 1/216 [00:02<07:45,  2.16s/it, loss=2.051, G=6.499, L=6.334, C=0.983, D=0.010, cf=0.000, lr=1.5e-07]


  [Conf] min=0.429 max=0.479 mean=0.454 active=12.0/12


Epoch 1 (Stage 1):  47%|████▋     | 101/216 [01:31<01:27,  1.31it/s, loss=1.505, G=6.117, L=1.920, C=0.403, D=0.027, cf=0.000, lr=1.6e-05]

...

Epoch 35 (Stage 2):  93%|█████████▎| 201/216 [02:27<00:10,  1.38it/s, loss=0.501, G=0.471, L=0.014, C=0.001, D=0.013, cf=0.000, lr=1.0e-06]


  [Conf] min=0.007 max=0.993 mean=0.304 active=5.9/12


Validation: 100%|██████████| 33/33 [03:59<00:00,  7.25s/it]



Epoch 35 Summary:
  Train Loss: 0.4624
    Global: 0.4320
    Local: 0.0125
    Consistency: 0.0009
    Diversity: 0.0142
    Conf Floor: 0.0007
  Val Loss: 0.9229
Final model saved: outputs\ablations\weak_grounding\clip4cad_gfa_final.pt

Training Complete!

ABLATION weak_grounding COMPLETE!
Checkpoints saved to: outputs\ablations\weak_grounding


WindowsPath('outputs/ablations/weak_grounding')

---
---
# PHASE 2: SELF-GROUNDING TRAINING

After Phase 1 (text-grounding) training completes, run Phase 2 to train self-grounding queries.

**Benefits:**
- Fast training (~30 min vs 4 hours for Phase 1)
- Only ~3K trainable parameters (12 queries × 256 dim)
- Enables text-free retrieval at inference time

**Prerequisites:**
- Phase 1 training must be complete (checkpoint_epoch35.pt exists)
- Data must be loaded (run cells 1-3)

---

In [ ]:
# =============================================================================
# PHASE 2: Self-Grounding Training Helper Function
# =============================================================================

from torch.utils.data import DataLoader
from clip4cad.data.gfa_dataset import gfa_collate_fn
from clip4cad.training.self_grounding_trainer import SelfGroundingTrainer

# Phase 2 config
PHASE2_CONFIG = {
    "num_epochs": 15,
    "learning_rate": 1e-3,  # Higher LR since only ~3K params
    "batch_size": 512,
    "validate_every": 5,
    "save_every": 5,
}


def train_self_grounding(ablation_type: str, phase1_checkpoint: str = None,
                         num_epochs: int = None, lr: float = None):
    """
    Phase 2: Train self-grounding queries on frozen Phase 1 model.
    
    Args:
        ablation_type: e.g., "asymmetric_grounding"
        phase1_checkpoint: Path to Phase 1 checkpoint (default: checkpoint_epoch35.pt)
        num_epochs: Number of Phase 2 epochs (default: 15)
        lr: Learning rate (default: 1e-3)
    
    Returns:
        output_dir: Path to output directory with Phase 2 checkpoint
    """
    print("\n" + "=" * 70)
    print(f"PHASE 2: SELF-GROUNDING TRAINING")
    print(f"Ablation: {ablation_type}")
    print("=" * 70)
    
    # Defaults
    if num_epochs is None:
        num_epochs = PHASE2_CONFIG["num_epochs"]
    if lr is None:
        lr = PHASE2_CONFIG["learning_rate"]
    
    # Output directory
    output_dir = Path(PATHS["output_base"]) / ablation_type
    
    # Find Phase 1 checkpoint
    if phase1_checkpoint is None:
        phase1_checkpoint = output_dir / "checkpoint_epoch35.pt"
    else:
        phase1_checkpoint = Path(phase1_checkpoint)
    
    if not phase1_checkpoint.exists():
        print(f"ERROR: Phase 1 checkpoint not found: {phase1_checkpoint}")
        print("Please run Phase 1 training first!")
        return None
    
    print(f"Phase 1 checkpoint: {phase1_checkpoint.name}")
    print(f"Epochs: {num_epochs}")
    print(f"Learning rate: {lr}")
    
    # Load Phase 1 checkpoint
    print("\nLoading Phase 1 model...")
    ckpt = torch.load(phase1_checkpoint, map_location=device, weights_only=False)
    config = get_ablation_config(PATHS["config_path"], ablation_type)
    
    model = CLIP4CAD_GFA_Ablation(config)
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    if missing:
        print(f"  Missing keys (random init): {len(missing)}")
    
    # Create dataloaders (using already-loaded datasets)
    train_loader = DataLoader(
        train_dataset,
        batch_size=PHASE2_CONFIG["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=gfa_collate_fn,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=PHASE2_CONFIG["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=gfa_collate_fn,
    )
    
    # Create trainer (freezes all except self_ground_queries)
    trainer = SelfGroundingTrainer(
        model=model,
        config=OmegaConf.to_container(config),
        device=device,
        output_dir=str(output_dir),
    )
    
    # Train Phase 2
    history = trainer.train(
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=num_epochs,
        lr=lr,
        validate_every=PHASE2_CONFIG["validate_every"],
        save_every=PHASE2_CONFIG["save_every"],
    )
    
    print("\n" + "=" * 70)
    print(f"PHASE 2 COMPLETE!")
    print(f"Checkpoint saved to: {output_dir / 'checkpoint_phase2_final.pt'}")
    print("=" * 70)
    
    # Cleanup
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    
    return output_dir


print("Phase 2 helper function defined.")
print("Usage: train_self_grounding('asymmetric_grounding')")

---
## Phase 2: ASYMMETRIC GROUNDING

Train self-grounding queries on the frozen asymmetric_grounding Phase 1 model.

In [ ]:
# === PHASE 2: ASYMMETRIC GROUNDING ===
# Prerequisites: Phase 1 training must be complete (checkpoint_epoch35.pt exists)
train_self_grounding("asymmetric_grounding", num_epochs=15)

---
## Phase 2: WEAK GROUNDING

Train self-grounding queries on the frozen weak_grounding Phase 1 model.

In [ ]:
# === PHASE 2: WEAK GROUNDING ===
train_self_grounding("weak_grounding", num_epochs=15)

---
## Phase 2: BASELINE

Train self-grounding queries on the frozen baseline Phase 1 model.

In [ ]:
# === PHASE 2: BASELINE ===
train_self_grounding("baseline", num_epochs=15)

---
## Phase 2: NO CONSISTENCY

Train self-grounding queries on the frozen no_consistency Phase 1 model.

In [ ]:
# === PHASE 2: NO CONSISTENCY ===
train_self_grounding("no_consistency", num_epochs=15)

---
## Phase 2: GLOBAL ONLY

Train self-grounding queries on the frozen global_only Phase 1 model.

In [ ]:
# === PHASE 2: GLOBAL ONLY ===
train_self_grounding("global_only", num_epochs=15)

---
## Phase 2: NO CONFIDENCE

Train self-grounding queries on the frozen no_confidence Phase 1 model.

In [ ]:
# === PHASE 2: NO CONFIDENCE ===
train_self_grounding("no_confidence", num_epochs=15)

---
---
# EVALUATION & COMPARISON

---

## Evaluate All Ablations

Compare retrieval metrics across all trained ablations.

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from clip4cad.data.gfa_dataset import gfa_collate_fn
import json

def evaluate_ablation(ablation_type: str, checkpoint: str = "best"):
    """Evaluate a trained ablation on validation set."""
    output_dir = Path(PATHS["output_base"]) / ablation_type
    
    if checkpoint == "best":
        ckpt_path = output_dir / "checkpoint_best.pt"
    else:
        ckpt_path = output_dir / f"checkpoint_epoch{checkpoint}.pt"
    
    if not ckpt_path.exists():
        print(f"Checkpoint not found: {ckpt_path}")
        return None
    
    print(f"\nEvaluating {ablation_type} from {ckpt_path.name}...")
    
    # Load model
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    config = get_ablation_config(PATHS["config_path"], ablation_type)
    
    model = CLIP4CAD_GFA_Ablation(config)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device).eval()
    
    # Create val loader
    val_loader = DataLoader(
        val_dataset, batch_size=64, shuffle=False,
        num_workers=0, pin_memory=True, collate_fn=gfa_collate_fn
    )
    
    # Generate embeddings
    text_emb, brep_emb, ids = [], [], []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="  Embeddings", leave=False):
            ids.extend(batch["sample_id"])
            outputs = model(batch)
            text_emb.append(F.normalize(outputs["z_text"].float(), p=2, dim=-1).cpu())
            brep_emb.append(F.normalize(outputs["z_brep"].float(), p=2, dim=-1).cpu())
    
    text_emb = torch.cat(text_emb)
    brep_emb = torch.cat(brep_emb)
    
    # Compute Text->BRep retrieval
    sim = torch.mm(text_emb, brep_emb.T)
    _, rankings = torch.topk(sim, k=10, dim=1)
    
    # Compute metrics
    n = len(ids)
    results = {}
    for k in [1, 5, 10]:
        hits = 0
        for i in range(n):
            for idx in rankings[i, :k]:
                if str(ids[idx]) == str(ids[i]):
                    hits += 1
                    break
        results[f"R@{k}"] = hits / n
    
    # Cleanup
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return results


# Evaluate all ablations
print("=" * 70)
print("ABLATION COMPARISON: Text->BRep Retrieval on Validation Set")
print("=" * 70)

all_results = {}
for abl in ["baseline", "no_consistency", "global_only", "no_confidence"]:
    results = evaluate_ablation(abl)
    if results:
        all_results[abl] = results

# Print comparison table
print("\n" + "=" * 70)
print(f"{'Ablation':<20} {'R@1':>10} {'R@5':>10} {'R@10':>10}")
print("-" * 50)
for abl, res in all_results.items():
    print(f"{abl:<20} {res['R@1']:>10.4f} {res['R@5']:>10.4f} {res['R@10']:>10.4f}")
print("=" * 70)

# Save results
results_path = Path(PATHS["output_base"]) / "ablation_comparison.json"
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to: {results_path}")

---
## Evaluate Single Ablation (All Checkpoints)

Evaluate one ablation across all its checkpoints to see training progression.

In [26]:
# =============================================================================
# SINGLE ABLATION CHECKPOINT-WISE EVALUATION
# =============================================================================
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from clip4cad.data.gfa_dataset import gfa_collate_fn
import json

# -----------------------------------------------------------------------------
# CONFIGURE: Which ablation to evaluate
# -----------------------------------------------------------------------------
EVAL_ABLATION = "weak_grounding"  # Change this: "baseline", "no_consistency", "global_only", "no_confidence"
EVAL_SPLIT = "val"  # "val" or "test"

# Specify which checkpoints to evaluate
# Options:
#   None or "all" - evaluate all checkpoints
#   [1, 5, 10, 15] - list of epoch numbers
#   ["best"] - only best checkpoint
#   [1, 5, "best"] - mix of epochs and "best"
EVAL_CHECKPOINTS = [ 15, 20, 35,"best"]  # Customize this list

# -----------------------------------------------------------------------------

def evaluate_checkpoint(ckpt_path, ablation_type, dataset):
    """Evaluate a single checkpoint with full metrics."""
    
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    config = get_ablation_config(PATHS["config_path"], ablation_type)
    
    model = CLIP4CAD_GFA_Ablation(config)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device).eval()
    
    # Info
    epoch = ckpt.get("epoch", "?")
    stage = ckpt.get("stage", "?")
    
    # Create loader
    loader = DataLoader(
        dataset, batch_size=64, shuffle=False,
        num_workers=0, pin_memory=True, collate_fn=gfa_collate_fn
    )
    
    # Generate embeddings
    text_emb, brep_emb, pc_emb, ids = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  Epoch {epoch}", leave=False):
            ids.extend(batch["sample_id"])
            outputs = model(batch)
            text_emb.append(F.normalize(outputs["z_text"].float(), p=2, dim=-1).cpu())
            if "z_brep" in outputs:
                brep_emb.append(F.normalize(outputs["z_brep"].float(), p=2, dim=-1).cpu())
            if "z_pc" in outputs:
                pc_emb.append(F.normalize(outputs["z_pc"].float(), p=2, dim=-1).cpu())
    
    text_emb = torch.cat(text_emb)
    brep_emb = torch.cat(brep_emb) if brep_emb else None
    pc_emb = torch.cat(pc_emb) if pc_emb else None
    
    # Compute metrics for all tasks
    k_values = [1, 5, 10]
    results = {"epoch": epoch, "stage": stage}
    
    tasks = [
        ("Text→BRep", text_emb, brep_emb),
        ("Text→PC", text_emb, pc_emb),
        ("PC→BRep", pc_emb, brep_emb),
        ("BRep→PC", brep_emb, pc_emb),
    ]
    
    for task_name, query_emb, gallery_emb in tasks:
        if query_emb is None or gallery_emb is None:
            continue
        
        # Compute similarities
        sim = torch.mm(query_emb, gallery_emb.T)
        _, rankings = torch.topk(sim, k=max(k_values), dim=1)
        
        # Compute R@k and mAP@k
        n = len(ids)
        task_results = {}
        
        for k in k_values:
            hits = 0
            ap_sum = 0.0
            
            for i in range(n):
                qid = str(ids[i])
                for rank, idx in enumerate(rankings[i, :k]):
                    if str(ids[idx]) == qid:
                        hits += 1
                        ap_sum += 1.0 / (rank + 1)
                        break
            
            task_results[f"R@{k}"] = hits / n
            task_results[f"mAP@{k}"] = ap_sum / n
        
        results[task_name] = task_results
    
    # Cleanup
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return results


# -----------------------------------------------------------------------------
# Find and filter checkpoints
# -----------------------------------------------------------------------------
output_dir = Path(PATHS["output_base"]) / EVAL_ABLATION

# Find all checkpoints
all_checkpoints = sorted(output_dir.glob("checkpoint_epoch*.pt"), 
                         key=lambda x: int(x.stem.split("epoch")[1]))

# Filter based on EVAL_CHECKPOINTS config
if EVAL_CHECKPOINTS is None or EVAL_CHECKPOINTS == "all":
    checkpoints = all_checkpoints
    # Add best checkpoint if exists
    best_ckpt = output_dir / "checkpoint_best.pt"
    if best_ckpt.exists() and best_ckpt not in checkpoints:
        checkpoints.append(best_ckpt)
else:
    checkpoints = []
    for spec in EVAL_CHECKPOINTS:
        if spec == "best":
            best_ckpt = output_dir / "checkpoint_best.pt"
            if best_ckpt.exists():
                checkpoints.append(best_ckpt)
        elif isinstance(spec, int):
            ckpt = output_dir / f"checkpoint_epoch{spec}.pt"
            if ckpt.exists():
                checkpoints.append(ckpt)
            else:
                print(f"Warning: checkpoint_epoch{spec}.pt not found")

print("=" * 80)
print(f"CHECKPOINT-WISE EVALUATION: {EVAL_ABLATION}")
print(f"Split: {EVAL_SPLIT}")
print(f"Found {len(checkpoints)} checkpoints to evaluate")
print("=" * 80)

if not checkpoints:
    print(f"No matching checkpoints found in {output_dir}")
else:
    # Use val_dataset or load test if needed
    if EVAL_SPLIT == "val":
        eval_dataset = val_dataset
    else:
        # Load test dataset
        eval_dataset = GFAMappedDataset(
            data_root=PATHS["data_root"],
            split="test",
            pc_file=PATHS["pc_file"],
            text_file=PATHS["text_file"],
            brep_file=PATHS["brep_file"],
            num_rotations=1,
            load_to_memory=False,
        )
    
    all_ckpt_results = []
    
    for ckpt_path in checkpoints:
        print(f"\nEvaluating: {ckpt_path.name}")
        results = evaluate_checkpoint(ckpt_path, EVAL_ABLATION, eval_dataset)
        all_ckpt_results.append(results)
        
        # Print summary
        if "Text→PC" in results:
            tb = results["Text→PC"]
            print(f"  Text→BRep: R@1={tb['R@1']:.4f}, R@5={tb['R@5']:.4f}, R@10={tb['R@10']:.4f}")
            print(f"             mAP@1={tb['mAP@1']:.4f}, mAP@5={tb['mAP@5']:.4f}, mAP@10={tb['mAP@10']:.4f}")
    
    # -----------------------------------------------------------------------------
    # Summary Table
    # -----------------------------------------------------------------------------
    print("\n" + "=" * 80)
    print(f"SUMMARY: {EVAL_ABLATION} on {EVAL_SPLIT}")
    print("=" * 80)
    print(f"{'Checkpoint':<25} {'Stage':<6} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'mAP@1':>8} {'mAP@5':>8}")
    print("-" * 80)
    
    for res in all_ckpt_results:
        epoch = res["epoch"]
        stage = res["stage"]
        name = f"epoch_{epoch}" if isinstance(epoch, int) else str(epoch)
        
        if "Text→BRep" in res:
            tb = res["Text→BRep"]
            print(f"{name:<25} {stage:<6} {tb['R@1']:>8.4f} {tb['R@5']:>8.4f} {tb['R@10']:>8.4f} "
                  f"{tb['mAP@1']:>8.4f} {tb['mAP@5']:>8.4f}")
    
    print("=" * 80)
    
    # Save detailed results
    results_path = output_dir / f"checkpoint_eval_{EVAL_SPLIT}.json"
    with open(results_path, "w") as f:
        json.dump(all_ckpt_results, f, indent=2)
    print(f"\nDetailed results saved to: {results_path}")

CHECKPOINT-WISE EVALUATION: weak_grounding
Split: val
Found 4 checkpoints to evaluate

Evaluating: checkpoint_epoch15.pt
  [Ablation] Using LEARNED confidence


  Epoch 15:   0%|          | 0/260 [00:00<?, ?it/s]

  Text→BRep: R@1=0.4696, R@5=0.7788, R@10=0.8692
             mAP@1=0.4696, mAP@5=0.5885, mAP@10=0.6008

Evaluating: checkpoint_epoch20.pt
  [Ablation] Using LEARNED confidence


  Epoch 20:   0%|          | 0/260 [00:00<?, ?it/s]

  Text→BRep: R@1=0.5281, R@5=0.8333, R@10=0.9053
             mAP@1=0.5281, mAP@5=0.6477, mAP@10=0.6576

Evaluating: checkpoint_epoch35.pt
  [Ablation] Using LEARNED confidence


  Epoch 35:   0%|          | 0/260 [00:00<?, ?it/s]

  Text→BRep: R@1=0.5966, R@5=0.8754, R@10=0.9300
             mAP@1=0.5966, mAP@5=0.7081, mAP@10=0.7156

Evaluating: checkpoint_best.pt
  [Ablation] Using LEARNED confidence


  Epoch 15:   0%|          | 0/260 [00:00<?, ?it/s]

  Text→BRep: R@1=0.4696, R@5=0.7788, R@10=0.8692
             mAP@1=0.4696, mAP@5=0.5885, mAP@10=0.6008

SUMMARY: weak_grounding on val
Checkpoint                Stage       R@1      R@5     R@10    mAP@1    mAP@5
--------------------------------------------------------------------------------
epoch_15                  1        0.4780   0.7438   0.8170   0.4780   0.5806
epoch_20                  2        0.5107   0.7759   0.8457   0.5107   0.6140
epoch_35                  2        0.5478   0.8008   0.8640   0.5478   0.6476
epoch_15                  1        0.4780   0.7438   0.8170   0.4780   0.5806

Detailed results saved to: outputs\ablations\weak_grounding\checkpoint_eval_val.json


In [27]:
# -----------------------------------------------------------------------------                                                                                                                                                              # Summary Table - ALL TASKS
# -----------------------------------------------------------------------------                                                                                                                                                              print("\n" + "=" * 120)
print(f"SUMMARY: {EVAL_ABLATION} on {EVAL_SPLIT}")
print("=" * 120)

# Header
print(f"{'Epoch':<8} {'Stage':<6} │ {'Text→BRep':^24} │ {'Text→PC':^24} │ {'PC→BRep':^24}")
print(f"{'':8} {'':6} │ {'R@1':>8} {'R@5':>8} {'R@10':>8} │ {'R@1':>8} {'R@5':>8} {'R@10':>8} │ {'R@1':>8} {'R@5':>8} {'R@10':>8}")
print("-" * 120)

for res in all_ckpt_results:
      epoch = res["epoch"]
      stage = res["stage"]

      # Get metrics for each task (default to 0 if not present)
      tb = res.get("Text→BRep", {})
      tp = res.get("Text→PC", {})
      pb = res.get("PC→BRep", {})

      print(f"{epoch:<8} {stage:<6} │ "
            f"{tb.get('R@1', 0):>8.4f} {tb.get('R@5', 0):>8.4f} {tb.get('R@10', 0):>8.4f} │ "
            f"{tp.get('R@1', 0):>8.4f} {tp.get('R@5', 0):>8.4f} {tp.get('R@10', 0):>8.4f} │ "
            f"{pb.get('R@1', 0):>8.4f} {pb.get('R@5', 0):>8.4f} {pb.get('R@10', 0):>8.4f}")

print("=" * 120)

# Also print mAP table
print(f"\n{'Epoch':<8} {'Stage':<6} │ {'Text→BRep mAP':^24} │ {'Text→PC mAP':^24} │ {'PC→BRep mAP':^24}")
print(f"{'':8} {'':6} │ {'@1':>8} {'@5':>8} {'@10':>8} │ {'@1':>8} {'@5':>8} {'@10':>8} │ {'@1':>8} {'@5':>8} {'@10':>8}")
print("-" * 120)

for res in all_ckpt_results:
      epoch = res["epoch"]
      stage = res["stage"]

      tb = res.get("Text→BRep", {})
      tp = res.get("Text→PC", {})
      pb = res.get("PC→BRep", {})

      print(f"{epoch:<8} {stage:<6} │ "
            f"{tb.get('mAP@1', 0):>8.4f} {tb.get('mAP@5', 0):>8.4f} {tb.get('mAP@10', 0):>8.4f} │ "
            f"{tp.get('mAP@1', 0):>8.4f} {tp.get('mAP@5', 0):>8.4f} {tp.get('mAP@10', 0):>8.4f} │ "
            f"{pb.get('mAP@1', 0):>8.4f} {pb.get('mAP@5', 0):>8.4f} {pb.get('mAP@10', 0):>8.4f}")

print("=" * 120)

SUMMARY: weak_grounding on val
Epoch    Stage  │        Text→BRep         │         Text→PC          │         PC→BRep         
                │      R@1      R@5     R@10 │      R@1      R@5     R@10 │      R@1      R@5     R@10
------------------------------------------------------------------------------------------------------------------------
15       1      │   0.4780   0.7438   0.8170 │   0.4696   0.7788   0.8692 │   0.2444   0.5246   0.6451
20       2      │   0.5107   0.7759   0.8457 │   0.5281   0.8333   0.9053 │   0.2720   0.5695   0.6864
35       2      │   0.5478   0.8008   0.8640 │   0.5966   0.8754   0.9300 │   0.2944   0.6043   0.7280
15       1      │   0.4780   0.7438   0.8170 │   0.4696   0.7788   0.8692 │   0.2444   0.5246   0.6451

Epoch    Stage  │      Text→BRep mAP       │       Text→PC mAP        │       PC→BRep mAP       
                │       @1       @5      @10 │       @1       @5      @10 │       @1       @5      @10
-----------------------------------

---
## Plot Checkpoint Progression

Visualize how metrics improve across checkpoints for a single ablation.

In [ ]:
# Plot checkpoint progression (run after the evaluation cell above)
import matplotlib.pyplot as plt

if 'all_ckpt_results' in dir() and all_ckpt_results:
    epochs = [r["epoch"] for r in all_ckpt_results]
    
    # Extract metrics
    r1 = [r["Text→BRep"]["R@1"] for r in all_ckpt_results if "Text→BRep" in r]
    r5 = [r["Text→BRep"]["R@5"] for r in all_ckpt_results if "Text→BRep" in r]
    r10 = [r["Text→BRep"]["R@10"] for r in all_ckpt_results if "Text→BRep" in r]
    map1 = [r["Text→BRep"]["mAP@1"] for r in all_ckpt_results if "Text→BRep" in r]
    map5 = [r["Text→BRep"]["mAP@5"] for r in all_ckpt_results if "Text→BRep" in r]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Recall plot
    ax = axes[0]
    ax.plot(epochs[:len(r1)], r1, 'o-', label='R@1', linewidth=2, markersize=8)
    ax.plot(epochs[:len(r5)], r5, 's-', label='R@5', linewidth=2, markersize=8)
    ax.plot(epochs[:len(r10)], r10, '^-', label='R@10', linewidth=2, markersize=8)
    ax.axvline(x=TRAIN_CONFIG["num_epochs_stage1"], color="gray", linestyle="--", alpha=0.5, label="Stage 2")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Recall", fontsize=12)
    ax.set_title(f"{EVAL_ABLATION}: Recall@k vs Epoch", fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    
    # mAP plot
    ax = axes[1]
    ax.plot(epochs[:len(map1)], map1, 'o-', label='mAP@1', linewidth=2, markersize=8)
    ax.plot(epochs[:len(map5)], map5, 's-', label='mAP@5', linewidth=2, markersize=8)
    ax.axvline(x=TRAIN_CONFIG["num_epochs_stage1"], color="gray", linestyle="--", alpha=0.5, label="Stage 2")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("mAP", fontsize=12)
    ax.set_title(f"{EVAL_ABLATION}: mAP@k vs Epoch", fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    
    # Save
    plot_path = Path(PATHS["output_base"]) / EVAL_ABLATION / f"checkpoint_progression_{EVAL_SPLIT}.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Saved: {plot_path}")
else:
    print("Run the checkpoint evaluation cell first!")

---
## Plot Training Curves

Compare training loss curves across ablations.

In [ ]:
import matplotlib.pyplot as plt
import json

# Load training histories
histories = {}
for abl in ["baseline", "no_consistency", "global_only", "no_confidence"]:
    history_path = Path(PATHS["output_base"]) / abl / "training_history.json"
    if history_path.exists():
        with open(history_path) as f:
            histories[abl] = json.load(f)
        print(f"Loaded history for {abl}")

if not histories:
    print("No training histories found. Train some ablations first!")
else:
    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = {"baseline": "blue", "no_consistency": "orange", 
              "global_only": "green", "no_confidence": "red"}

    metrics = [
        ("train_loss", "Total Loss"),
        ("train_global", "Global Loss"),
        ("train_local", "Local Loss"),
        ("train_consist", "Consistency Loss"),
    ]

    for idx, (key, title) in enumerate(metrics):
        ax = axes[idx // 2, idx % 2]
        for abl, hist in histories.items():
            if key in hist and hist[key]:
                ax.plot(hist[key], label=abl, color=colors.get(abl, "gray"))
        ax.axvline(x=TRAIN_CONFIG["num_epochs_stage1"], color="gray", 
                   linestyle="--", alpha=0.5, label="Stage 2")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(PATHS["output_base"]) / "training_curves.png", dpi=150)
    plt.show()
    print(f"\nSaved: {Path(PATHS['output_base']) / 'training_curves.png'}")

---
## Memory Check

In [ ]:
import psutil

gc.collect()
torch.cuda.empty_cache()

process = psutil.Process()
ram_gb = process.memory_info().rss / 1e9
available_ram = psutil.virtual_memory().available / 1e9

print(f"Process RAM: {ram_gb:.1f} GB")
print(f"Available RAM: {available_ram:.1f} GB")

if torch.cuda.is_available():
    gpu_alloc = torch.cuda.memory_allocated() / 1e9
    print(f"GPU Allocated: {gpu_alloc:.1f} GB")